# Network Intrusion Triage (UNSW-NB15) — Comprehensive Workflow

This notebook provides an interactive walkthrough of the **Network Intrusion Triage** pipeline. Following repository standards:
- All configuration is loaded from `configs/config.yaml` via `src.config`.
- All data loading, preprocessing, EDA, and modeling logic is imported from `src/` (zero duplicated code).
- `src/pipeline.py` remains the single authoritative producer of saved artifacts (`models/best_model.joblib`, `reports/metrics.json`, and `reports/figures/*.png`).

In [ ]:
import json
from pathlib import Path
import sys

from IPython.display import Image, display
import joblib
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))

from src.config import load_config, resolve_path
from src.data_loader import load_train_test
from src.pipeline import run_pipeline
from src.preprocessing import prepare_features

## 1. Configuration & Dataset Audit
Load the centralized configuration (`configs/config.yaml`) and inspect the UNSW-NB15 Parquet splits: dimensions, missing values, duplicates, and target class balance (`label` and `attack_cat`).

In [ ]:
config = load_config()
train_df, test_df = load_train_test(config)

audit_summary = pd.DataFrame({
    'Split': ['Train', 'Test'],
    'Rows': [len(train_df), len(test_df)],
    'Columns': [train_df.shape[1], test_df.shape[1]],
    'Missing Values': [int(train_df.isna().sum().sum()), int(test_df.isna().sum().sum())],
    'Exact Duplicates': [int(train_df.duplicated().sum()), int(test_df.duplicated().sum())],
    'Suspicious Rate (%)': [
        round(train_df['label'].mean() * 100, 2),
        round(test_df['label'].mean() * 100, 2),
    ],
})
display(audit_summary)
train_df.head(3)

## 2. Leakage-Free Feature Preparation Check
Verify that `prepare_features()` excludes `attack_cat` (which encodes the attack family and would leak `label` 1:1) and `label`, leaving 34 clean input features (3 categorical + 31 numeric).

In [ ]:
X_train_check, y_train_check = prepare_features(train_df, config, require_target=True)
assert 'attack_cat' not in X_train_check.columns
assert 'label' not in X_train_check.columns
print(f'Prepared feature matrix shape: {X_train_check.shape} | Target length: {len(y_train_check)}')

## 3. Exploratory Data Analysis (EDA) Visuals
Run `src.pipeline.run_pipeline()` (or inspect the generated figures in `reports/figures/`) to examine class imbalance, service-level attack rates, log-scaled flow distributions, and top numeric feature correlations.

In [ ]:
summary = run_pipeline()
figures_dir = resolve_path(config['paths']['figures_dir'])

eda_captions = {
    'eda_class_distribution.png': 'Binary target imbalance and attack_cat breakdown.',
    'eda_service_attack_rate.png': 'Suspicious traffic rate across the top 10 network services.',
    'eda_numeric_distributions.png': 'Log-scaled dur, sbytes, dbytes, and rate distributions by class.',
    'eda_correlation_heatmap.png': 'Top 10 numeric features most correlated with label.',
}
for fig_name, caption in eda_captions.items():
    print(f'{fig_name}: {caption}')
    display(Image(filename=str(figures_dir / fig_name)))

## 4. Experiment Comparison & Evaluation
Compare `LogisticRegression` (baseline), `RandomForestClassifier`, and `HistGradientBoostingClassifier` on the unseen test split, focusing on **Recall** (minimizing missed attacks) and **F1-Score** (balancing alert fatigue).

In [ ]:
print('Best Model Selected (by priority metric):', summary['best_model'])
metrics_df = pd.DataFrame(summary['results']).T[
    ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'confusion_matrix']
]
display(metrics_df)

for eval_fig in ['model_comparison.png', 'confusion_matrix_best.png', 'roc_curves.png']:
    print(eval_fig)
    display(Image(filename=str(figures_dir / eval_fig)))

## 5. Inference Smoke Test on Unseen Flows
Load the persisted `models/best_model.joblib` pipeline and score 5 unseen test flows to demonstrate SOC triage output (`predicted_label`, `suspicious_prob`).

In [ ]:
best_model = joblib.load(resolve_path(config['paths']['best_model']))
sample_raw = test_df.sample(5, random_state=config['project']['seed'])
X_sample, y_sample = prepare_features(sample_raw, config, require_target=True)

triage_preview = pd.DataFrame({
    'proto': sample_raw['proto'].astype(str).values,
    'service': sample_raw['service'].astype(str).values,
    'true_label': y_sample.values,
    'predicted_label': best_model.predict(X_sample),
    'suspicious_prob': best_model.predict_proba(X_sample)[:, 1].round(4),
})
display(triage_preview)